# Implementing a CNN and investigating convolution with JAX
**COMP6701 · CNN project**

My goal in this notebook is to build the models, understand how they work, and run the main project experiments. I explain every line of code with a short English comment directly above it. I use the Python standard library, NumPy, Matplotlib, and JAX. I write the layers, cross-entropy loss, Adam optimizer, and training loop directly. JAX provides the convolution primitive and automatic differentiation, which the project allows.

The tables, plots, and checkpoints come from running the notebook. There are no prefilled accuracy results. I use `quick` mode to check the workflow and understand the code. For the final experiments, I use `full` mode and discuss the limitations in my report. Runtime depends on the device, and the first steps take longer because JAX compiles the computations.

### Opening and running the notebook

1. **Colab:** I open the `.ipynb` file with Upload notebook and run the cells in order. The core experiments can run on a CPU; a GPU can help with the larger runs.
2. **Kaggle:** I import the file as a notebook and enable Internet for automatic dataset downloads. For an offline run, I attach the two datasets in separate folders and set `LOCAL_DATA_DIRS`.
3. To run the full protocol, I change `MODE = "quick"` to `MODE = "full"` in the settings cell. On Colab, I download the outputs before the session ends. On Kaggle, the outputs are written under `/kaggle/working`.
4. The last code cell creates a ZIP containing the results, checkpoints, CSV files, figures, and reproduction instructions. I also save the notebook separately.

I use this notebook to study, run, and develop the project. My final report needs the actual results, my analysis, code references, and each team member's contributions. Running the experiments and interpreting them are still part of the work.

**Development validation — October 21, 2026:** All quick-mode cells were run in a local Python environment on a CPU with JAX 0.12.0. This covered 12 training runs on real MNIST and Fashion-MNIST data, 144 robustness evaluations, figure and ZIP creation, correctness checks, and checks that the saved checkpoints matched the lowest validation loss. Full mode and the Kaggle/Colab interfaces were not tested in that validation. Saved cell outputs are cleared so that new results come from the current run.

## 1. Connecting the implementation to Lab 2

| Lab 2 component | Implementation here | Why I use it |
|---|---|---|
| `nn.Linear` and `nn.ReLU` | `dense` and `relu` using arrays | To write out the layer calculations |
| `nn.Conv2d` | `conv2d` using a JAX primitive | To control the weights, bias, and data shapes |
| `nn.MaxPool2d` | `pool2d` | To compare max pooling, average pooling, and direct subsampling |
| `CrossEntropyLoss` | Stable log-softmax and explicit cross-entropy | To make the loss calculation clear |
| `torch.optim.Adam` | `adam_init` and `adam_update` | To write out the parameter update |
| `backward()` | `jax.value_and_grad` | To use the automatic differentiation allowed by the project |
| Training and test loops | `train_model` and `evaluate` | To keep training, validation, and testing separate |
| MNIST only | MNIST and Fashion-MNIST | To cover both required datasets |
| One convolution, BatchNorm, and Dropout | Two convolutions, without BatchNorm or Dropout | To use a LeNet-style model and reduce other factors in the comparison |
| Selecting the lowest test error | Selecting the lowest validation loss | To keep the test set out of checkpoint selection |

### Models and experiment questions

- `mlp`: My fully connected baseline. I choose its hidden-layer width from the CNN's parameter budget.
- `cnn_max`: My main LeNet-style model, with two convolution and max-pooling blocks.
- `cnn_avg`: The same weight shapes and parameter count, with average pooling in place of max pooling.
- `cnn_subsample`: The same output dimensions, but I keep just one value from each 2-by-2 block. This removes local aggregation while keeping downsampling.
- `cnn_max_gap` and `cnn_no_pool_gap`: Two models with the same Global Average Pooling (GAP) head. The first has intermediate pooling, and the second does not. This lets me remove intermediate pooling while keeping the parameter count fixed. Both models still average spatial positions in the head. Feature-map size, receptive field, and computation cost change as part of this comparison.

Comparing a CNN and an MLP with similar parameter counts gives me an architecture comparison. It does not separate every effect of locality and weight sharing. I also examine weight sharing through numerical checks, filter shapes, and a translation check. Residual connections are an optional extension and are outside the scope of this version.

## 2. Setting up the libraries

If JAX is already installed, this cell keeps the existing installation. If JAX is missing, it installs the CPU version. GPU execution needs a JAX installation that is compatible with the environment's drivers. I print the detected devices to check which hardware is available.

In [ ]:
# I use this to check which packages are already installed.
import importlib.util
# I use subprocess to install any missing packages.
import subprocess
# I use sys to run pip with the current Python interpreter.
import sys
# I check the three packages this notebook needs.
for package in ["numpy", "matplotlib", "jax"]:
    # I only install a package if it is missing.
    if importlib.util.find_spec(package) is None:
        # For a fresh JAX install, I use the CPU package.
        requirement = "jax[cpu]" if package == "jax" else package
        # I install the missing package in the current environment.
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])
# I use Path to handle file and folder paths.
from pathlib import Path
# I use os for environment settings and file operations.
import os
# I ask JAX not to reserve most of the GPU memory at startup.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# I use gzip to open the compressed dataset files.
import gzip
# I use struct to read the binary IDX headers.
import struct
# I use hashes to check the downloaded files.
import hashlib
# I use urllib to download the datasets.
import urllib.request
# I use JSON to save settings and run details.
import json
# I use CSV files to save the result tables.
import csv
# I use this timer to measure how long each run takes.
import time
# I record the operating system information for reproducibility.
import platform
# I use metadata to get the installed package versions.
from importlib import metadata
# I use shutil to put the outputs into a ZIP file.
import shutil
# I record run timestamps in UTC.
from datetime import datetime, timezone
# I use NumPy for loading and preparing the data.
import numpy as np
# I use Matplotlib for the plots.
import matplotlib.pyplot as plt
# I use JAX for model calculations and automatic differentiation.
import jax
# I use jnp for array operations that JAX can differentiate.
import jax.numpy as jnp
# I use lax for the low-level convolution and pooling operations.
from jax import lax
# I print the JAX version and the devices it can use.
print("JAX", jax.__version__, "| devices:", jax.devices())
# I set a readable default size for the plots and text.
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

## 3. Experiment settings and output folders

In `quick` mode, I use 2,000 training examples, 500 validation examples, and 500 test examples per dataset, with three epochs and one seed. This is a preliminary run.

In `full` mode, I use 55,000 training examples, 5,000 validation examples, and all 10,000 test examples, with 15 epochs and three seeds. All models receive the same data, epoch budget, batch size, optimizer, and learning rate. These shared settings may not be optimal for every model. If I tune the models later, I need to use equal tuning budgets and the same validation set.

For robustness, I use a fixed test subset: 500 examples in quick mode and 2,000 in full mode. I always calculate the accuracy drop relative to the clean version of that same subset. For the training curves, I evaluate a fixed probe from the training set to keep the cost of each epoch manageable.

In [ ]:
# I start with quick mode; I can switch to full for the final experiments.
MODE = "quick"
# I can set separate local folders here if I already have the datasets.
LOCAL_DATA_DIRS = {"mnist": None, "fashion_mnist": None}
# I allow the notebook to download missing data files.
ALLOW_DOWNLOAD = True
# I list the six models I want to compare.
MODEL_NAMES = ["mlp", "cnn_max", "cnn_avg", "cnn_subsample", "cnn_max_gap", "cnn_no_pool_gap"]
# I use a small dataset and one seed for a quick test run.
QUICK = dict(train_n=2000, val_n=500, test_n=500, epochs=3, seeds=[0], robustness_n=500, train_probe_n=1000)
# I use the full split and three seeds for the larger experiment.
FULL = dict(train_n=55000, val_n=5000, test_n=10000, epochs=15, seeds=[0, 1, 2], robustness_n=2000, train_probe_n=5000)
# I check that the selected mode is valid.
assert MODE in {"quick", "full"}, "MODE must be quick or full"
# I copy the settings for the selected mode.
CFG = dict(QUICK if MODE == "quick" else FULL)
# I keep the training settings and evaluation seeds consistent across models.
CFG.update(mode=MODE, batch_size=128, learning_rate=0.001, split_seed=2026, eval_seed=3107, models=MODEL_NAMES, datasets=["mnist", "fashion_mnist"], noise_sigmas=[0.05, 0.10, 0.20])
# I choose a writable output location for Kaggle, Colab, or a local run.
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
# I keep downloaded data in a separate cache folder.
DATA_ROOT = BASE / "cnn_data"
# I make a short ID from the experiment settings.
CONFIG_ID = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:10]
# I add a UTC timestamp so separate runs do not overwrite each other.
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
# I give this run its own output folder.
OUT = BASE / "cnn_project_runs" / f"{MODE}_{CONFIG_ID}_{RUN_STAMP}"
# I create the output folder if it does not exist.
OUT.mkdir(parents=True, exist_ok=True)
# I save the settings so I can repeat the experiment later.
(OUT / "config.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")
# I print the output location and the settings before starting.
print("Output:", OUT, "\nConfiguration:", CFG)

## 4. Loading MNIST and Fashion-MNIST without torchvision

An IDX file contains a header followed by an array of pixels or labels. I convert the pixels to `float32` values between zero and one. I use the **NHWC** image layout: number of images, height, width, and channels. This differs from the channel layout used by PyTorch in the lab.

The two datasets use the same filenames, so I keep their local folders separate. Each folder can contain subfolders and either `.gz` files or unpacked IDX files. I check the official compressed files against their checksums. If downloading fails, the notebook raises an error instead of silently using synthetic data.

In [ ]:
# I list the four standard IDX files needed for each dataset.
IDX_FILES = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]
# I keep the official MNIST checksums to check the downloads.
MNIST_MD5 = ["f68b3c2dcbeaaa9fbdd348bbdeb94873", "d53e105ee54ea40749a09fcbcd1e9432", "9fb629c4189551a2d022fa330f9573f3", "ec29112dd5afa0611ce80d1b7f02629c"]
# I keep the official Fashion-MNIST checksums as well.
FASHION_MD5 = ["8d4fb7e6c68d591d4c3dfef9ec88bf0d", "25c81989df183df01b3e8a0aad5dffbe", "bef4ecab320f06d8554ea6380940ec79", "bb300cfdad3c16e7a12a480ee83cd310"]
# I link each dataset to its download URL and checksums.
DATA_SOURCES = {"mnist": ("https://storage.googleapis.com/cvdf-datasets/mnist/", MNIST_MD5), "fashion_mnist": ("https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/", FASHION_MD5)}
# I keep the Fashion-MNIST class names here for reference.
FASHION_LABELS = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
# I use this helper to find a data file locally or download it.
def get_idx_file(dataset, filename, expected_md5):
    # I check whether a local folder was provided for this dataset.
    local = LOCAL_DATA_DIRS[dataset]
    # If I have a local folder, I look there first.
    if local is not None:
        # I look for either the compressed file or the uncompressed IDX file.
        candidates = list(Path(local).rglob(filename)) + list(Path(local).rglob(filename[:-3]))
        # I check whether the expected file was found.
        if not candidates:
            # I stop with a clear message if the local IDX file is missing.
            raise FileNotFoundError(f"Missing {filename} under {local}; provide the official IDX files, not CSV.")
        # I prefer the compressed file when both versions are available.
        path = sorted(candidates, key=lambda p: (p.suffix != ".gz", str(p)))[0]
    # Otherwise, I use the download cache.
    else:
        # I build the cache path for this dataset and file.
        path = DATA_ROOT / dataset / filename
        # I create the dataset folder if needed.
        path.parent.mkdir(parents=True, exist_ok=True)
        # I only download the file if it is not already cached.
        if not path.exists():
            # I check whether downloading is allowed.
            if not ALLOW_DOWNLOAD:
                # I explain how to provide the data when downloading is disabled.
                raise FileNotFoundError("Enable Internet/ALLOW_DOWNLOAD or set LOCAL_DATA_DIRS to dataset folders.")
            # I build the full URL for this file.
            url = DATA_SOURCES[dataset][0] + filename
            # I print which file is being downloaded.
            print("Downloading", dataset, filename, flush=True)
            # I create a download request with a simple user-agent header.
            request = urllib.request.Request(url, headers={"User-Agent": "CNN-course-project/1.0"})
            # I open the URL with a timeout so a request cannot wait forever.
            with urllib.request.urlopen(request, timeout=120) as response:
                # I read the downloaded bytes.
                payload = response.read()
            # I check the checksum before saving the downloaded file.
            assert hashlib.md5(payload).hexdigest() == expected_md5, f"Checksum mismatch: {filename}"
            # I save the verified file in the cache.
            path.write_bytes(payload)
    # I also verify compressed files that were already available locally.
    if path.suffix == ".gz":
        # I stop if the gzip file does not match the official checksum.
        assert hashlib.md5(path.read_bytes()).hexdigest() == expected_md5, f"Invalid official gzip file: {path}"
    # I return the path to the verified data file.
    return path

# I use this helper to turn an IDX file into a NumPy array.
def read_idx(path):
    # I choose the right reader for a gzip file or a regular file.
    opener = gzip.open if path.suffix == ".gz" else open
    # I open the file in binary mode.
    with opener(path, "rb") as handle:
        # I read its contents into memory.
        payload = handle.read()
    # I read the IDX header fields, including the data type and number of dimensions.
    zero1, zero2, dtype_code, ndim = struct.unpack(">BBBB", payload[:4])
    # I check that this is the expected unsigned-byte IDX format.
    assert (zero1, zero2, dtype_code) == (0, 0, 8) and ndim in (1, 3), "Unexpected IDX header"
    # I read the size of each dimension from the header.
    shape = struct.unpack(">" + "I" * ndim, payload[4:4 + 4 * ndim])
    # I read the bytes after the header as unsigned integers.
    values = np.frombuffer(payload, dtype=np.uint8, offset=4 + 4 * ndim)
    # I check that the number of values matches the declared shape.
    assert values.size == int(np.prod(shape)), "Truncated or malformed IDX file"
    # I reshape the values and return an independent copy.
    return values.reshape(shape).copy()

# I use stratified sampling to keep class proportions close to the original data.
def stratified_indices(labels, count, seed):
    # I check that the requested sample size is available.
    assert 0 < count <= len(labels)
    # I create a random generator with a fixed seed.
    rng = np.random.default_rng(seed)
    # I find the classes present in the labels.
    classes = np.unique(labels)
    # I count how many samples belong to each class.
    sizes = np.array([np.sum(labels == label) for label in classes])
    # I calculate the proportional sample quota for each class.
    exact = sizes / len(labels) * count
    # I start with the whole-number part of each quota.
    quotas = np.floor(exact).astype(int)
    # I count how many samples still need to be allocated.
    remainder = count - int(quotas.sum())
    # I assign the remaining samples to the largest fractional quotas.
    quotas[np.argsort(-(exact - quotas), kind="stable")[:remainder]] += 1
    # I prepare a list for the selected indices.
    chosen = []
    # I sample each class according to its quota.
    for label, quota in zip(classes, quotas):
        # I shuffle the indices of this class and keep the required number.
        chosen.extend(rng.permutation(np.flatnonzero(labels == label))[:quota].tolist())
    # I shuffle the combined selection before returning it.
    return rng.permutation(np.asarray(chosen, dtype=np.int64))

# I use this function to prepare one dataset for the experiments.
def load_dataset(name):
    # I get the checksums for this dataset.
    checksums = DATA_SOURCES[name][1]
    # I find or download all four data files.
    paths = [get_idx_file(name, f, h) for f, h in zip(IDX_FILES, checksums)]
    # I load the official training and test images and labels.
    raw_train, y_all, raw_test, y_official_test = [read_idx(p) for p in paths]
    # I check the expected image counts and dimensions.
    assert raw_train.shape == (60000, 28, 28) and raw_test.shape == (10000, 28, 28)
    # I check the label counts and confirm that both sets contain classes 0 to 9.
    assert y_all.shape == (60000,) and y_official_test.shape == (10000,) and set(np.unique(y_all)) == set(range(10)) and set(np.unique(y_official_test)) == set(range(10))
    # I select validation samples only from the official training set.
    val_idx = stratified_indices(y_all, CFG["val_n"], CFG["split_seed"])
    # I remove the validation indices from the training pool.
    remaining = np.setdiff1d(np.arange(len(y_all)), val_idx)
    # I select training samples from the remaining pool.
    train_idx = remaining[stratified_indices(y_all[remaining], CFG["train_n"], CFG["split_seed"] + 1)]
    # I select the test subset separately from the official test set.
    test_idx = stratified_indices(y_official_test, CFG["test_n"], CFG["split_seed"] + 2)
    # I check that training and validation do not share any samples.
    assert np.intersect1d(train_idx, val_idx).size == 0
    # I save the exact split indices so the same split can be reproduced.
    np.savez_compressed(OUT / f"{name}_split_indices.npz", train=train_idx, validation=val_idx, test=test_idx)
    # I convert pixels to float32 in [0, 1] and add the channel dimension.
    convert = lambda array: array.astype(np.float32)[..., None] / np.float32(255.0)
    # I build the training, validation, and test arrays.
    data = dict(x_train=convert(raw_train[train_idx]), y_train=y_all[train_idx].astype(np.int32), x_val=convert(raw_train[val_idx]), y_val=y_all[val_idx].astype(np.int32), x_test=convert(raw_test[test_idx]), y_test=y_official_test[test_idx].astype(np.int32))
    # I choose a fixed training subset to monitor during training.
    data["probe_idx"] = stratified_indices(data["y_train"], min(CFG["train_probe_n"], CFG["train_n"]), CFG["split_seed"] + 3)
    # I choose a fixed test subset for the robustness experiments.
    data["robust_idx"] = stratified_indices(data["y_test"], min(CFG["robustness_n"], CFG["test_n"]), CFG["eval_seed"])
    # I save the indices of both evaluation subsets.
    np.savez_compressed(OUT / f"{name}_evaluation_indices.npz", train_probe=data["probe_idx"], robustness=data["robust_idx"])
    # I calculate hashes of the actual files used in this run.
    manifest = [{"filename": p.name, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in paths]
    # I save the data-file hashes with the outputs.
    (OUT / f"{name}_data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    # I print the sizes of the prepared splits.
    print(name, {key: value.shape for key, value in data.items() if key.startswith("x_")}, flush=True)
    # I return the prepared dataset.
    return data

In [ ]:
# I load both datasets using the same experiment settings.
DATA = {name: load_dataset(name) for name in CFG["datasets"]}
# I display a few training images from each dataset.
for name, data in DATA.items():
    # I create one row of ten image panels.
    fig, axes = plt.subplots(1, 10, figsize=(14, 2))
    # I fill each panel with one training example.
    for index, ax in enumerate(axes):
        # I display the single-channel image in grayscale.
        ax.imshow(data["x_train"][index, :, :, 0], cmap="gray", vmin=0, vmax=1)
        # I show its true label above the image.
        ax.set_title(str(int(data["y_train"][index])))
        # I hide the axes to make the image easier to see.
        ax.axis("off")
    # I add the dataset name to the figure.
    fig.suptitle(name + " | training examples")
    # I adjust the spacing between the panels.
    fig.tight_layout()
    # I save the example images with the other outputs.
    fig.savefig(OUT / f"{name}_examples.png", bbox_inches="tight")
    # I display the figure in the notebook.
    plt.show()

## 5. Writing the layers

For a dense layer, I calculate `x @ W + b`. For a convolution, I apply the same kernel at every spatial position. As in most machine-learning libraries, the exact operation is **cross-correlation** because the filter is not flipped. I use the low-level JAX primitive for the parallel computation and check its output against an independent NumPy implementation later.

The main model has this sequence of shapes:

`28×28×1 → Conv5:24×24×6 → Pool2:12×12×6 → Conv5:8×8×16 → Pool2:4×4×16 → 256 → 120 → 84 → 10`.

In [ ]:
# I define ReLU directly using an elementwise maximum.
def relu(x):
    # I replace negative values with zero and keep positive values.
    return jnp.maximum(x, 0.0)

# I define a fully connected layer.
def dense(x, params):
    # I multiply the input by the weights and add the bias.
    return x @ params["w"] + params["b"]

# I define a 2D convolution with stride 1 and no padding.
def conv2d(x, params):
    # I use NHWC inputs and HWIO kernels in JAX's convolution primitive.
    result = lax.conv_general_dilated(x, params["w"], window_strides=(1, 1), padding="VALID", dimension_numbers=("NHWC", "HWIO", "NHWC"))
    # I add one shared bias per output channel.
    return result + params["b"]

# I put the pooling options and their controls in one function.
def pool2d(x, kind="max"):
    # I check whether intermediate pooling is disabled.
    if kind == "none":
        # I return the feature map without changing it.
        return x
    # I check whether this is the subsampling control.
    if kind == "subsample":
        # I take every other row and column without combining nearby values.
        return x[:, ::2, ::2, :]
    # I use a 2 by 2 window and leave the batch and channel axes unchanged.
    window = (1, 2, 2, 1)
    # I move the pooling window by two pixels each time.
    strides = (1, 2, 2, 1)
    # I handle max pooling here.
    if kind == "max":
        # I take the largest value in each window.
        return lax.reduce_window(x, -jnp.inf, lax.max, window, strides, "VALID")
    # I handle average pooling here.
    if kind == "avg":
        # I add the four values in each window and divide by four.
        return lax.reduce_window(x, 0.0, lax.add, window, strides, "VALID") / 4.0
    # I raise an error if the pooling option is not recognised.
    raise ValueError(f"Unknown pooling kind: {kind}")

# I use this helper to initialise a layer's weights and biases.
def init_layer(key, shape, fan_in):
    # I use He scaling for the random weights because the network uses ReLU.
    weights = jax.random.normal(key, shape, dtype=jnp.float32) * jnp.sqrt(2.0 / fan_in)
    # I return the weights and initialise one bias per output to zero.
    return {"w": weights, "b": jnp.zeros((shape[-1],), dtype=jnp.float32)}

# I use this helper to count trainable parameters.
def parameter_count(params):
    # I add up the sizes of all arrays in the parameter tree.
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params))

# I record the pooling type and head type for each CNN variant.
CNN_SPECS = {"cnn_max": ("max", False), "cnn_avg": ("avg", False), "cnn_subsample": ("subsample", False), "cnn_max_gap": ("max", True), "cnn_no_pool_gap": ("none", True)}
# I calculate the parameter budget of the main CNN from its layer sizes.
CNN_TARGET = (5 * 5 * 1 * 6 + 6) + (5 * 5 * 6 * 16 + 16) + (256 * 120 + 120) + (120 * 84 + 84) + (84 * 10 + 10)
# I choose the MLP width to get a similar parameter count.
MLP_WIDTH = int(round((CNN_TARGET - 10) / (784 + 1 + 10)))

# I initialise the requested model using a reproducible seed.
def init_model(model_name, seed):
    # I split the seed into a separate random key for each layer.
    keys = jax.random.split(jax.random.PRNGKey(seed), 5)
    # I initialise the fully connected baseline in this branch.
    if model_name == "mlp":
        # I create the MLP's hidden layer.
        fc1 = init_layer(keys[0], (784, MLP_WIDTH), 784)
        # I create the MLP's ten-class output layer.
        fc2 = init_layer(keys[1], (MLP_WIDTH, 10), MLP_WIDTH)
        # I return the two MLP layers in one dictionary.
        return {"fc1": fc1, "fc2": fc2}
    # I get the pooling and head settings for this CNN.
    pool_kind, use_gap = CNN_SPECS[model_name]
    # I use 16 head inputs with global averaging, or 256 with flattening.
    head_input = 16 if use_gap else 256
    # I start a dictionary for the CNN's parameters.
    params = {}
    # I create six 5 by 5 filters for the one-channel input.
    params["conv1"] = init_layer(keys[0], (5, 5, 1, 6), 25)
    # I create sixteen 5 by 5 filters, each connected to all six input channels.
    params["conv2"] = init_layer(keys[1], (5, 5, 6, 16), 150)
    # I create the first dense layer with 120 outputs.
    params["fc1"] = init_layer(keys[2], (head_input, 120), head_input)
    # I create the second dense layer with 84 outputs.
    params["fc2"] = init_layer(keys[3], (120, 84), 120)
    # I create the final layer with one output per class.
    params["fc3"] = init_layer(keys[4], (84, 10), 84)
    # I return the parameters for all five layers.
    return params

# I define the forward pass and an option to return intermediate features.
def forward(params, x, model_name, return_features=False):
    # I use a separate forward path for the MLP.
    if model_name == "mlp":
        # I flatten each image into a vector of 784 pixels.
        flat = x.reshape((x.shape[0], -1))
        # I apply the hidden dense layer followed by ReLU.
        hidden = relu(dense(flat, params["fc1"]))
        # I calculate the ten raw class scores.
        logits = dense(hidden, params["fc2"])
        # I return the hidden features too if they were requested.
        return (logits, {"hidden": hidden}) if return_features else logits
    # I get this CNN's pooling and head settings.
    pool_kind, use_gap = CNN_SPECS[model_name]
    # I apply the first convolution and ReLU.
    conv1 = relu(conv2d(x, params["conv1"]))
    # I apply the selected pooling operation or its control.
    pool1 = pool2d(conv1, pool_kind)
    # I apply the second convolution and ReLU.
    conv2 = relu(conv2d(pool1, params["conv2"]))
    # I apply the second pooling operation or its control.
    pool2 = pool2d(conv2, pool_kind)
    # I either average over space or flatten the feature maps for the head.
    flat = jnp.mean(pool2, axis=(1, 2)) if use_gap else pool2.reshape((x.shape[0], -1))
    # I apply the 120-unit dense layer and ReLU.
    hidden1 = relu(dense(flat, params["fc1"]))
    # I apply the 84-unit dense layer and ReLU.
    hidden2 = relu(dense(hidden1, params["fc2"]))
    # I calculate the final ten class scores.
    logits = dense(hidden2, params["fc3"])
    # I collect the intermediate feature maps for visualisation.
    features = {"conv1_relu": conv1, "pool1": pool1, "conv2_relu": conv2, "pool2": pool2}
    # I return either the scores alone or the scores with the feature maps.
    return (logits, features) if return_features else logits

## 6. Writing the loss and Adam optimizer

I first calculate a stable log-softmax for each image. Cross-entropy is the negative log-probability of the correct class. I subtract the largest logit before taking exponentials to avoid overflow.

In Adam, `m` is a moving average of the gradients and `v` is a moving average of the squared gradients. Bias correction accounts for both averages starting at zero. I use the same learning rate for all models. The `mask` keeps padded examples in the last batch out of the loss calculation.

In [ ]:
# I define a numerically stable log-softmax.
def log_softmax(logits):
    # I subtract the largest score in each row before taking exponentials.
    shifted = logits - jnp.max(logits, axis=-1, keepdims=True)
    # I subtract the log normalising factor to get log-probabilities.
    return shifted - jnp.log(jnp.sum(jnp.exp(shifted), axis=-1, keepdims=True))

# I calculate the statistics needed for loss and accuracy.
def batch_statistics(logits, labels, mask):
    # I convert the class scores into log-probabilities.
    log_probs = log_softmax(logits)
    # I take the negative log-probability of each sample's true class.
    losses = -log_probs[jnp.arange(labels.shape[0]), labels]
    # I predict the class with the highest score.
    predictions = jnp.argmax(logits, axis=-1)
    # I sum the losses, correct predictions, and real sample count using the mask.
    return jnp.sum(losses * mask), jnp.sum((predictions == labels) * mask), jnp.sum(mask)

# I define the scalar loss used to train a model.
def model_loss(params, x, y, mask, model_name):
    # I get the model's raw output scores.
    logits = forward(params, x, model_name)
    # I calculate the total loss and number of real samples.
    loss_sum, correct, count = batch_statistics(logits, y, mask)
    # I divide by the real sample count to get the mean loss.
    return loss_sum / jnp.maximum(count, 1.0)

# I initialise the Adam optimiser state.
def adam_init(params):
    # I make zero arrays with the same structure as the parameters.
    zeros = jax.tree_util.tree_map(jnp.zeros_like, params)
    # I start the update counter and both moving averages at zero.
    return {"step": jnp.array(0, dtype=jnp.int32), "m": zeros, "v": zeros}

# I implement one Adam update directly.
def adam_update(params, grads, state, learning_rate, beta1=0.9, beta2=0.999, epsilon=1e-8):
    # I increase the update counter by one.
    step = state["step"] + 1
    # I update the moving average of the gradients.
    m = jax.tree_util.tree_map(lambda old, g: beta1 * old + (1.0 - beta1) * g, state["m"], grads)
    # I update the moving average of the squared gradients.
    v = jax.tree_util.tree_map(lambda old, g: beta2 * old + (1.0 - beta2) * g * g, state["v"], grads)
    # I correct the first moving average for its zero initialisation.
    m_hat = jax.tree_util.tree_map(lambda value: value / (1.0 - beta1 ** step), m)
    # I apply the same bias correction to the second moving average.
    v_hat = jax.tree_util.tree_map(lambda value: value / (1.0 - beta2 ** step), v)
    # I update each parameter using the corrected averages.
    updated = jax.tree_util.tree_map(lambda p, first, second: p - learning_rate * first / (jnp.sqrt(second) + epsilon), params, m_hat, v_hat)
    # I return the new parameters and optimiser state.
    return updated, {"step": step, "m": m, "v": v}

# I build the training and evaluation functions for one architecture.
def make_operations(model_name):
    # I define one training step for this model.
    def train_step(params, state, x, y, mask, learning_rate):
        # I use automatic differentiation to get the loss and parameter gradients.
        value, grads = jax.value_and_grad(model_loss)(params, x, y, mask, model_name)
        # I apply one Adam update to the parameters.
        updated, new_state = adam_update(params, grads, state, learning_rate)
        # I return the updated state and the loss before the update.
        return updated, new_state, value
    # I define an evaluation step that does not update the model.
    def evaluation_step(params, x, y, mask):
        # I return the batch statistics from the current predictions.
        return batch_statistics(forward(params, x, model_name), y, mask)
    # I JIT-compile both functions to speed up repeated calls.
    return jax.jit(train_step), jax.jit(evaluation_step)

# I create these functions once per model and reuse them across seeds.
OPERATIONS = {name: make_operations(name) for name in MODEL_NAMES}

## 7. Checking correctness before training

These checks test the calculations directly. I compare convolution with an independent NumPy implementation, pooling with a hand-calculated answer, a gradient with finite differences, and an Adam step with the update formula. I also check that every architecture produces the expected output shape and finite gradients.

The translation check tests convolution without pooling and only uses the interior region. It checks **equivariance**: shifting the input shifts the filter response. This does not mean the final classifier output stays unchanged. I measure that behavior in the robustness experiments.

In [ ]:
# I use a small NumPy convolution as an independent reference for testing.
def reference_conv_numpy(x, weights, bias):
    # I read the input dimensions.
    n, height, width, channels = x.shape
    # I read the kernel dimensions and channel counts.
    kh, kw, input_channels, output_channels = weights.shape
    # I check that the input and kernel channel counts match.
    assert channels == input_channels
    # I allocate the output array for a valid convolution.
    output = np.zeros((n, height - kh + 1, width - kw + 1, output_channels), dtype=np.float32)
    # I move through the output rows.
    for row in range(output.shape[1]):
        # I move through the output columns.
        for col in range(output.shape[2]):
            # I select the image patch under the kernel at this position.
            patch = x[:, row:row + kh, col:col + kw, :]
            # I multiply each patch by the same kernel weights, sum, and add the bias.
            output[:, row, col, :] = np.einsum("nhwc,hwco->no", patch, weights) + bias
    # I return the reference convolution output.
    return output

# I group the correctness checks in one function.
def run_correctness_checks():
    # I keep a list of checks that have passed.
    passed = []
    # I use a fixed seed for the small test arrays.
    rng = np.random.default_rng(17)
    # I create two small images with two channels each.
    x = rng.normal(size=(2, 6, 7, 2)).astype(np.float32)
    # I create a small kernel with three output channels.
    w = rng.normal(size=(2, 3, 2, 3)).astype(np.float32)
    # I create one bias per output channel.
    b = rng.normal(size=(3,)).astype(np.float32)
    # I compare the JAX convolution with the independent NumPy result.
    np.testing.assert_allclose(np.asarray(conv2d(jnp.array(x), {"w": jnp.array(w), "b": jnp.array(b)})), reference_conv_numpy(x, w, b), rtol=2e-5, atol=2e-5)
    # I record that the convolution reference check passed.
    passed.append("Convolution matches independent NumPy reference")
    # I make a 4 by 4 grid with values from 0 to 15.
    grid = jnp.arange(16, dtype=jnp.float32).reshape(1, 4, 4, 1)
    # I compare max pooling with the expected values calculated by hand.
    np.testing.assert_allclose(np.asarray(pool2d(grid, "max")).ravel(), [5, 7, 13, 15])
    # I compare average pooling with the expected means.
    np.testing.assert_allclose(np.asarray(pool2d(grid, "avg")).ravel(), [2.5, 4.5, 10.5, 12.5])
    # I check which four values the subsampling control selects.
    np.testing.assert_allclose(np.asarray(pool2d(grid, "subsample")).ravel(), [0, 2, 8, 10])
    # I record that the pooling checks passed.
    passed.append("Max/average pooling and subsampling match hand calculations")
    # I use uniform class scores and mask out the second example.
    stats = batch_statistics(jnp.zeros((2, 10)), jnp.array([2, 7]), jnp.array([1.0, 0.0]))
    # I check that the loss for one uniform ten-class prediction is log(10).
    np.testing.assert_allclose(float(stats[0]), np.log(10), rtol=1e-6)
    # I check that the masked example does not count as a real sample.
    assert float(stats[2]) == 1.0
    # I check that log-softmax stays finite for very large scores.
    assert np.all(np.isfinite(np.asarray(log_softmax(jnp.array([[10000.0, -10000.0, 0.0]])))))
    # I record that the loss and masking checks passed.
    passed.append("Stable cross-entropy and padding mask")
    # I define a simple kernel-dependent function for checking gradients.
    objective = lambda kernel: jnp.mean(conv2d(jnp.array(x), {"w": kernel, "b": jnp.array(b)}) ** 2)
    # I calculate its gradient with JAX.
    analytic = np.asarray(jax.grad(objective)(jnp.array(w)))
    # I choose a small step for the finite-difference check.
    epsilon = 1e-3
    # I check three different kernel entries.
    for index in [(0, 0, 0, 0), (1, 2, 1, 2), (0, 1, 1, 1)]:
        # I copy the weights for the positive perturbation.
        plus = w.copy()
        # I slightly increase the selected weight.
        plus[index] += epsilon
        # I copy the weights for the negative perturbation.
        minus = w.copy()
        # I slightly decrease the selected weight.
        minus[index] -= epsilon
        # I estimate the derivative using a central difference.
        numeric = (float(objective(jnp.array(plus))) - float(objective(jnp.array(minus)))) / (2 * epsilon)
        # I compare the automatic gradient with the numerical estimate.
        np.testing.assert_allclose(analytic[index], numeric, rtol=0.02, atol=0.005)
    # I record that the gradient check passed.
    passed.append("Convolution gradient matches finite differences")
    # I create two simple parameters for the Adam check.
    p = {"w": jnp.array([1.0, -2.0])}
    # I specify two known gradients.
    g = {"w": jnp.array([0.2, -0.4])}
    # I apply the first Adam update.
    updated, state = adam_update(p, g, adam_init(p), 0.01)
    # I compare the result with the direct formula for the first corrected step.
    np.testing.assert_allclose(np.asarray(updated["w"]), np.asarray(p["w"] - 0.01 * g["w"] / (jnp.abs(g["w"]) + 1e-8)), atol=1e-6)
    # I check that Adam's update counter is now one.
    assert int(state["step"]) == 1
    # I record that the Adam check passed.
    passed.append("Adam first update matches the bias-corrected formula")
    # I shift the test images one pixel right and fill the left border with zeros.
    shifted = np.pad(x, ((0, 0), (0, 0), (1, 0), (0, 0)))[:, :, :x.shape[2], :]
    # I calculate the convolution response to the original images.
    original_response = conv2d(jnp.array(x), {"w": jnp.array(w), "b": jnp.array(b)})
    # I calculate the response to the shifted images using the same weights.
    shifted_response = conv2d(jnp.array(shifted), {"w": jnp.array(w), "b": jnp.array(b)})
    # I check that the interior responses shift with the input.
    np.testing.assert_allclose(np.asarray(shifted_response[:, :, 1:, :]), np.asarray(original_response[:, :, :-1, :]), rtol=2e-5, atol=2e-5)
    # I record the shared-weight translation-equivariance check.
    passed.append("Shared convolution is translation-equivariant away from boundaries")
    # I make a small random batch with the usual image shape.
    toy_x = jnp.array(rng.uniform(size=(2, 28, 28, 1)).astype(np.float32))
    # I check every model architecture.
    for name in MODEL_NAMES:
        # I initialise this model with a fixed seed.
        params = init_model(name, 0)
        # I run its forward pass.
        logits = forward(params, toy_x, name)
        # I check that it returns ten finite scores for each of the two images.
        assert logits.shape == (2, 10) and np.all(np.isfinite(np.asarray(logits)))
        # I calculate the loss gradients for all its parameters.
        gradients = jax.grad(model_loss)(params, toy_x, jnp.array([1, 2]), jnp.ones(2), name)
        # I check that none of the gradients contain NaN or infinity.
        assert all(np.all(np.isfinite(np.asarray(leaf))) for leaf in jax.tree_util.tree_leaves(gradients))
        # I print the model name and parameter count.
        print(name, "parameters:", parameter_count(params))
    # I check that the MLP and main CNN parameter counts differ by less than 1%.
    assert abs(parameter_count(init_model("mlp", 0)) - CNN_TARGET) / CNN_TARGET < 0.01
    # I check that the two GAP variants have exactly the same parameter count.
    assert parameter_count(init_model("cnn_max_gap", 0)) == parameter_count(init_model("cnn_no_pool_gap", 0))
    # I record that the architecture and parameter-budget checks passed.
    passed.append("All model shapes/gradients are valid; parameter budgets are matched")
    # I save the list of successful checks with the run outputs.
    (OUT / "correctness_checks.json").write_text(json.dumps({"passed": passed}, indent=2), encoding="utf-8")
    # I print the checks that passed.
    print("PASSED:", *passed, sep="\n- ")
    # I return the list of successful checks.
    return passed

# I run these checks before training the models.
CHECKS = run_correctness_checks()

## 8. Batches, evaluation, and checkpoint saving

I pad the last batch with zero images so JAX does not need to compile a different computation for a smaller batch. A mask keeps these extra examples out of the loss, accuracy, and gradients. The models do not use BatchNorm, so padding also does not change batch statistics for the real examples.

I calculate accuracy and loss from totals across the dataset, rather than taking an unweighted average of batch means. This gives the last batch the correct weight. I save checkpoints as NPZ files without pickle.

In [ ]:
# I create batches with padding and a mask for the final batch.
def batches(x, y, batch_size, order=None):
    # I use the original order unless a shuffled order was provided.
    indices = np.arange(len(y)) if order is None else order
    # I move through the indices one batch at a time.
    for start in range(0, len(indices), batch_size):
        # I select the indices for the current batch.
        ids = indices[start:start + batch_size]
        # I count the real examples in this batch.
        actual = len(ids)
        # I allocate a fixed-size image batch filled with zeros.
        bx = np.zeros((batch_size,) + x.shape[1:], dtype=np.float32)
        # I allocate the label array for this batch.
        by = np.zeros(batch_size, dtype=np.int32)
        # I start with a zero mask for all batch positions.
        mask = np.zeros(batch_size, dtype=np.float32)
        # I copy the real images into the start of the batch.
        bx[:actual] = x[ids]
        # I copy their labels into the matching positions.
        by[:actual] = y[ids]
        # I mark the real examples with ones in the mask.
        mask[:actual] = 1.0
        # I yield the images, labels, and mask together.
        yield bx, by, mask

# I evaluate a model on a given set of images and labels.
def evaluate(params, x, y, model_name):
    # I select this model's compiled evaluation function.
    evaluation_step = OPERATIONS[model_name][1]
    # I start totals for loss, correct predictions, and sample count.
    totals = np.zeros(3, dtype=np.float64)
    # I evaluate the data in batches.
    for bx, by, mask in batches(x, y, CFG["batch_size"]):
        # I get each batch's statistics without updating the parameters.
        stats = evaluation_step(params, jnp.asarray(bx), jnp.asarray(by), jnp.asarray(mask))
        # I bring the statistics back to the CPU and add them to the totals.
        totals += np.asarray(jax.device_get(stats), dtype=np.float64)
    # I calculate the overall mean loss and accuracy from the totals.
    return {"loss": float(totals[0] / totals[2]), "accuracy": float(totals[1] / totals[2]), "n": int(totals[2])}

# I use this helper to save a list of result rows as CSV.
def write_csv(path, rows):
    # I check whether there are any rows to save.
    if not rows:
        # I leave the function if the result list is empty.
        return
    # I convert the destination to a Path object.
    path = Path(path)
    # I write to a temporary file first to avoid leaving a partial CSV.
    temporary = path.with_name(path.name + ".tmp")
    # I open the temporary CSV file with UTF-8 encoding.
    with open(temporary, "w", newline="", encoding="utf-8") as handle:
        # I use the first row's keys as the column names.
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        # I write the header row.
        writer.writeheader()
        # I write all the result rows.
        writer.writerows(rows)
        # I flush any data still in the file buffer.
        handle.flush()
        # I finish writing the file before replacing the old version.
        os.fsync(handle.fileno())
    # I replace the destination with the completed temporary file.
    temporary.replace(path)

# I save the model parameters in an NPZ checkpoint.
def save_parameters(path, params):
    # I give each weight and bias array a name containing its layer name.
    arrays = {f"{layer}__{key}": np.asarray(value) for layer, items in params.items() for key, value in items.items()}
    # I save the arrays in a compressed file without pickle.
    np.savez_compressed(path, **arrays)

# I use this helper to load a saved checkpoint.
def load_parameters(path):
    # I start an empty parameter dictionary.
    params = {}
    # I open the NPZ archive with pickle disabled.
    with np.load(path, allow_pickle=False) as archive:
        # I read each array in the checkpoint.
        for name in archive.files:
            # I split the array name into its layer name and parameter key.
            layer, key = name.split("__", 1)
            # I rebuild the original parameter structure using JAX arrays.
            params.setdefault(layer, {})[key] = jnp.asarray(archive[name])
    # I return the restored parameters.
    return params

## 9. Training and selecting a checkpoint with validation

Each epoch uses every training example. For a given seed and epoch, all models receive the same batch order. Models with matching weight shapes also start with the same initial values.

After each epoch, I record loss and accuracy on the fixed training probe and the validation set. I select the checkpoint using the lowest **validation loss** and evaluate the test set after training. Recorded runtime includes JIT compilation and evaluation, so it is not a pure benchmark of architecture speed.

In [ ]:
# I train one model on one dataset with a specified seed.
def train_model(dataset_name, data, model_name, seed):
    # I give this model and seed their own output folder.
    run_dir = OUT / dataset_name / f"{model_name}_seed{seed}"
    # I create the folder for this run.
    run_dir.mkdir(parents=True, exist_ok=True)
    # I initialise the model with the selected seed.
    params = init_model(model_name, seed)
    # I initialise a fresh Adam state.
    state = adam_init(params)
    # I select the compiled training step for this architecture.
    train_step = OPERATIONS[model_name][0]
    # I start the best validation loss at infinity.
    best_val = float("inf")
    # I leave the best epoch unset until validation has been evaluated.
    best_epoch = 0
    # I prepare a list for the training history.
    history = []
    # I get the fixed training-probe indices.
    probe = data["probe_idx"]
    # I start timing this training run.
    started = time.perf_counter()
    # I train for the configured number of epochs.
    for epoch in range(1, CFG["epochs"] + 1):
        # I start a timer for this epoch.
        epoch_start = time.perf_counter()
        # I use the same shuffled order for models with the same seed and epoch.
        order = np.random.default_rng(100000 + seed * 1000 + epoch).permutation(len(data["y_train"]))
        # I reset the running training-loss total for this epoch.
        online_loss_sum = 0.0
        # I go through every training batch.
        for bx, by, mask in batches(data["x_train"], data["y_train"], CFG["batch_size"], order):
            # I calculate gradients and apply one Adam update.
            params, state, value = train_step(params, state, jnp.asarray(bx), jnp.asarray(by), jnp.asarray(mask), CFG["learning_rate"])
            # I weight the batch loss by its real sample count before adding it.
            online_loss_sum += float(value) * float(mask.sum())
        # I evaluate the current model on the fixed training probe.
        train_metrics = evaluate(params, data["x_train"][probe], data["y_train"][probe], model_name)
        # I evaluate the same model on the validation set.
        val_metrics = evaluate(params, data["x_val"], data["y_val"], model_name)
        # I check whether the validation loss is finite.
        if not np.isfinite(val_metrics["loss"]):
            # I stop with an error if training has produced an invalid loss.
            raise FloatingPointError(f"Non-finite loss in {dataset_name}/{model_name}/seed{seed}")
        # I collect this epoch's metrics and elapsed time in one row.
        row = dict(epoch=epoch, train_probe_loss=train_metrics["loss"], train_probe_accuracy=train_metrics["accuracy"], val_loss=val_metrics["loss"], val_accuracy=val_metrics["accuracy"], online_train_loss=online_loss_sum / len(data["y_train"]), epoch_seconds=time.perf_counter() - epoch_start)
        # I add the row to the training history.
        history.append(row)
        # I only select a new checkpoint when validation loss improves.
        if val_metrics["loss"] < best_val:
            # I update the best validation loss.
            best_val = val_metrics["loss"]
            # I record the epoch that produced this checkpoint.
            best_epoch = epoch
            # I save the selected model parameters.
            save_parameters(run_dir / "best_params.npz", params)
        # I save the history after every epoch in case the session ends early.
        write_csv(run_dir / "history.csv", history)
        # I print the training and validation progress for this epoch.
        print(f"{dataset_name:13s} {model_name:16s} seed={seed} epoch={epoch:02d} train_probe_acc={train_metrics['accuracy']:.3f} val_acc={val_metrics['accuracy']:.3f} val_loss={val_metrics['loss']:.4f}", flush=True)
    # I record the training time, including compilation and validation.
    training_seconds = time.perf_counter() - started
    # I reload the checkpoint selected by validation loss.
    best_params = load_parameters(run_dir / "best_params.npz")
    # I evaluate that checkpoint on the test set after training is finished.
    test_metrics = evaluate(best_params, data["x_test"], data["y_test"], model_name)
    # I collect the final metrics and model details in a summary.
    summary = dict(dataset=dataset_name, model=model_name, seed=seed, parameters=parameter_count(best_params), best_epoch=best_epoch, best_val_loss=best_val, test_loss=test_metrics["loss"], test_accuracy=test_metrics["accuracy"], test_n=test_metrics["n"], training_seconds_including_jit_and_validation=training_seconds)
    # I save the summary as JSON.
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    # I return the selected model, its history, and the run details.
    return {"params": best_params, "history": history, "summary": summary, "run_dir": run_dir}

## 10. Running all models on both datasets

Quick mode runs 12 short training jobs, and full mode runs 36. I do not assume the CNN must win: a negative result is still a result to record and explain. If the full run takes too long, I can use a GPU or change the budget and document that change in the report.

In [ ]:
# I keep the trained models and run details in this dictionary.
RUNS = {}
# I prepare a list for the results from all completed runs.
RESULT_ROWS = []
# I run the experiment on each dataset.
for dataset_name, data in DATA.items():
    # I train each of the selected architectures.
    for model_name in MODEL_NAMES:
        # I repeat training for each configured seed.
        for seed in CFG["seeds"]:
            # I train the model and evaluate its selected checkpoint.
            run = train_model(dataset_name, data, model_name, seed)
            # I store the run under its dataset, model, and seed.
            RUNS[(dataset_name, model_name, seed)] = run
            # I add this run's summary to the result table.
            RESULT_ROWS.append(run["summary"])
            # I save the results as each run finishes.
            write_csv(OUT / "results_per_seed.csv", RESULT_ROWS)
# I print where the completed training results were saved.
print("Training complete. Results saved to", OUT)

## 11. Accuracy, parameter counts, and learning curves

Accuracy values in the CSV files are fractions between zero and one. The printed tables show them as percentages. The `std` column describes variation across training seeds; it is not a confidence interval over the population of images. Quick mode has only one seed, so the standard deviation cannot be estimated and is shown as `NA`.

In [ ]:
# I use this helper to summarise results across training seeds.
def mean_std(values):
    # I convert the values to a floating-point array.
    values = np.asarray(values, dtype=float)
    # I return the mean and sample standard deviation, or None for a single seed.
    return float(values.mean()), (float(values.std(ddof=1)) if len(values) > 1 else None)

# I prepare the aggregated results table.
SUMMARY_ROWS = []
# I summarise each dataset separately.
for dataset_name in CFG["datasets"]:
    # I print a heading explaining that the displayed accuracy is a percentage.
    print("\n", dataset_name, "| accuracy mean +/- sample SD (%)")
    # I summarise one architecture at a time.
    for model_name in MODEL_NAMES:
        # I select the result rows for this dataset and model.
        rows = [r for r in RESULT_ROWS if r["dataset"] == dataset_name and r["model"] == model_name]
        # I calculate the mean accuracy and its variation across seeds.
        average, spread = mean_std([r["test_accuracy"] for r in rows])
        # I collect the model size, seed count, mean accuracy, and mean loss.
        summary = dict(dataset=dataset_name, model=model_name, parameters=rows[0]["parameters"], seeds=len(rows), test_accuracy_mean=average, test_accuracy_std=spread, test_loss_mean=float(np.mean([r["test_loss"] for r in rows])))
        # I add this model's summary to the table.
        SUMMARY_ROWS.append(summary)
        # I display NA when one seed is not enough to estimate variation.
        spread_text = "NA (one seed)" if spread is None else f"{100 * spread:.2f}"
        # I print the parameter count and accuracy summary.
        print(f"{model_name:18s} params={summary['parameters']:6d} accuracy={100 * average:.2f} +/- {spread_text}")
# I save the aggregated results as CSV.
write_csv(OUT / "results_summary.csv", SUMMARY_ROWS)

# I plot the learning curves separately for each dataset.
for dataset_name in CFG["datasets"]:
    # I make four panels for training-probe and validation loss and accuracy.
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    # I list the four metrics I want to plot.
    metrics = ["train_probe_loss", "val_loss", "train_probe_accuracy", "val_accuracy"]
    # I put each metric on its own axis.
    for ax, metric in zip(axes.ravel(), metrics):
        # I add a curve for every model.
        for model_name in MODEL_NAMES:
            # I collect this metric's history across all training seeds.
            values = np.asarray([[r[metric] for r in RUNS[(dataset_name, model_name, seed)]["history"]] for seed in CFG["seeds"]])
            # I calculate the mean value at each epoch.
            average = values.mean(axis=0)
            # I create the epoch numbers for the horizontal axis.
            epochs = np.arange(1, len(average) + 1)
            # I plot the average learning curve for this model.
            line, = ax.plot(epochs, average, label=model_name)
            # I only draw a variation band when there is more than one seed.
            if len(values) > 1:
                # I calculate the sample standard deviation at each epoch.
                spread = values.std(axis=0, ddof=1)
                # I shade one standard deviation on either side of the mean.
                ax.fill_between(epochs, average - spread, average + spread, color=line.get_color(), alpha=0.12)
        # I label the vertical axis with the metric name.
        ax.set_ylabel(metric.replace("_", " "))
        # I label the horizontal axis with the epoch number.
        ax.set_xlabel("Epoch")
        # I add a light grid to make the curves easier to read.
        ax.grid(alpha=0.25)
    # I add a legend identifying the models.
    axes[0, 0].legend(fontsize=8)
    # I include the dataset, run mode, and meaning of the shaded bands in the title.
    fig.suptitle(f"{dataset_name} | {MODE} | mean across seeds; shading = sample SD")
    # I adjust the figure layout.
    fig.tight_layout()
    # I save the learning-curve figure.
    fig.savefig(OUT / f"{dataset_name}_learning_curves.png", bbox_inches="tight")
    # I display the learning curves in the notebook.
    plt.show()

## 12. Testing translations and noise

I evaluate the validation-selected checkpoints on modified images without retraining or adding augmentation.

- **Translations:** I shift images by one and two pixels in each of four directions. Newly exposed pixels are zero, and pixels do not wrap around the image.
- **Noise:** I add Gaussian noise with standard deviations of 0.05, 0.10, and 0.20 on the zero-to-one pixel scale, then clip the result to that range.
- I use exactly the same images and noise for every model and training seed.
- Pooling does not guarantee robustness. In particular, downsampling can make a model sensitive to a one-pixel shift.

In [ ]:
# I shift images without wrapping pixels around the edges.
def translate_images(images, dx, dy):
    # I read the image height and width.
    height, width = images.shape[1:3]
    # I check that the shift is smaller than the image dimensions.
    assert abs(dx) < width and abs(dy) < height
    # I start with zero-filled output images.
    shifted = np.zeros_like(images)
    # I find the source rows that remain inside the shifted image.
    sy0, sy1 = max(0, -dy), min(height, height - dy)
    # I find the source columns that remain inside the shifted image.
    sx0, sx1 = max(0, -dx), min(width, width - dx)
    # I copy the valid image region into its shifted position.
    shifted[:, sy0 + dy:sy1 + dy, sx0 + dx:sx1 + dx, :] = images[:, sy0:sy1, sx0:sx1, :]
    # I return the shifted images with zero-filled borders.
    return shifted

# I generate the clean, translated, and noisy evaluation conditions.
def perturbations(images):
    # I yield the unchanged images first as the reference condition.
    yield {"condition": "clean", "kind": "clean", "severity": 0.0, "dx": 0, "dy": 0, "noise_seed": -1}, images
    # I test shifts of one and two pixels.
    for radius in [1, 2]:
        # I test the four directions at each shift size.
        for dx, dy in [(radius, 0), (-radius, 0), (0, radius), (0, -radius)]:
            # I record the direction and size of this shift.
            condition = dict(condition=f"shift_dx{dx}_dy{dy}", kind="translation", severity=float(radius), dx=dx, dy=dy, noise_seed=-1)
            # I yield the condition details and the shifted images.
            yield condition, translate_images(images, dx, dy)
    # I test each configured noise level.
    for index, sigma in enumerate(CFG["noise_sigmas"]):
        # I use a fixed seed for the noise at this level.
        noise_seed = CFG["eval_seed"] + 100 + index
        # I generate Gaussian noise with the selected standard deviation.
        noise = np.random.default_rng(noise_seed).normal(0.0, sigma, images.shape).astype(np.float32)
        # I add the noise and clip the pixels back to [0, 1].
        noisy = np.clip(images + noise, 0.0, 1.0).astype(np.float32)
        # I record the noise level and seed.
        condition = dict(condition=f"noise_sigma{sigma:.2f}", kind="noise", severity=float(sigma), dx=0, dy=0, noise_seed=noise_seed)
        # I yield the condition details and the noisy images.
        yield condition, noisy

# I make a small 3 by 3 image to check the shifting function.
SHIFT_TOY = np.arange(9, dtype=np.float32).reshape(1, 3, 3, 1)
# I check that a right shift gives the expected pixels and a zero-filled border.
np.testing.assert_array_equal(translate_images(SHIFT_TOY, 1, 0)[0, :, :, 0], [[0, 0, 1], [0, 3, 4], [0, 6, 7]])
# I check an upward shift against its expected output.
np.testing.assert_array_equal(translate_images(SHIFT_TOY, 0, -1)[0, :, :, 0], [[3, 4, 5], [6, 7, 8], [0, 0, 0]])
# I prepare a table for all robustness evaluations.
ROBUST_ROWS = []
# I evaluate robustness on each dataset.
for dataset_name, data in DATA.items():
    # I get the indices of the fixed robustness subset.
    ids = data["robust_idx"]
    # I select the clean images for this subset.
    clean_x = data["x_test"][ids]
    # I select their matching labels.
    labels = data["y_test"][ids]
    # I keep each model's clean-subset accuracy as its reference.
    clean_scores = {}
    # I generate each perturbed image batch once and share it across models.
    for condition, images in perturbations(clean_x):
        # I evaluate every architecture under the same condition.
        for model_name in MODEL_NAMES:
            # I evaluate each trained seed separately.
            for seed in CFG["seeds"]:
                # I get the checkpoint selected for this model and seed.
                params = RUNS[(dataset_name, model_name, seed)]["params"]
                # I measure its performance on the current images.
                metrics = evaluate(params, images, labels, model_name)
                # I check whether this is the clean reference condition.
                if condition["kind"] == "clean":
                    # I save this model's accuracy on the unchanged subset.
                    clean_scores[(model_name, seed)] = metrics["accuracy"]
                # I record the metrics and accuracy drop relative to the same clean subset.
                row = dict(dataset=dataset_name, model=model_name, seed=seed, **condition, n=metrics["n"], accuracy=metrics["accuracy"], loss=metrics["loss"], accuracy_drop=clean_scores[(model_name, seed)] - metrics["accuracy"])
                # I add this evaluation to the robustness table.
                ROBUST_ROWS.append(row)
        # I save the table after each condition is complete.
        write_csv(OUT / "robustness_per_seed.csv", ROBUST_ROWS)
        # I print which robustness condition has finished.
        print("Robustness:", dataset_name, condition["condition"], "completed", flush=True)

## 13. Summarizing and plotting robustness

For translations, I first average the four directions within each training seed. I then calculate the mean and standard deviation across seeds. I do not treat the four directions as four independent training runs. The vertical axis shows the accuracy drop in **percentage points**. A negative drop means accuracy was slightly higher under that condition.

In [ ]:
# I prepare a summary table for the robustness plots.
ROBUST_SUMMARY = []
# I draw separate robustness plots for each dataset.
for dataset_name in CFG["datasets"]:
    # I create one panel for shifts and one for noise.
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    # I handle translations and noise in separate panels.
    for ax, kind in zip(axes, ["translation", "noise"]):
        # I include the clean condition at level zero on each plot.
        levels = [0.0, 1.0, 2.0] if kind == "translation" else [0.0] + CFG["noise_sigmas"]
        # I add one curve per model.
        for model_name in MODEL_NAMES:
            # I prepare a list of mean accuracy drops.
            means = []
            # I prepare a list of standard deviations.
            deviations = []
            # I calculate the result at each perturbation level.
            for level in levels:
                # I collect one average drop per training seed.
                seed_drops = []
                # I process the results for each seed separately.
                for seed in CFG["seeds"]:
                    # I use clean results at zero and the selected perturbation type otherwise.
                    selected_kind = "clean" if level == 0 else kind
                    # I select the matching dataset, model, seed, and severity rows.
                    selected = [r for r in ROBUST_ROWS if r["dataset"] == dataset_name and r["model"] == model_name and r["seed"] == seed and r["kind"] == selected_kind and r["severity"] == level]
                    # I average over directions within the seed before comparing seeds.
                    seed_drops.append(float(np.mean([r["accuracy_drop"] for r in selected])))
                # I calculate the mean drop and its variation across training seeds.
                average, spread = mean_std(seed_drops)
                # I add this result to the robustness summary table.
                ROBUST_SUMMARY.append(dict(dataset=dataset_name, model=model_name, kind=kind, severity=level, accuracy_drop_mean=average, accuracy_drop_std=spread, seeds=len(seed_drops)))
                # I convert the mean drop to percentage points for the plot.
                means.append(100 * average)
                # I use no error-bar length for a single seed; its table entry stays missing.
                deviations.append(0.0 if spread is None else 100 * spread)
            # I plot the mean drops with their error bars.
            ax.errorbar(levels, means, yerr=deviations, marker="o", capsize=3, label=model_name)
        # I label the horizontal axis with the shift size or noise level.
        ax.set_xlabel("Translation radius (pixels; average of 4 directions)" if kind == "translation" else "Gaussian noise standard deviation")
        # I label the accuracy drop in percentage points.
        ax.set_ylabel("Accuracy drop (percentage points)")
        # I add a light grid.
        ax.grid(alpha=0.25)
    # I add a legend for the models.
    axes[0].legend(fontsize=8)
    # I put the dataset and run mode in the title.
    fig.suptitle(f"{dataset_name} | robustness | {MODE}")
    # I adjust the plot spacing.
    fig.tight_layout()
    # I save the robustness figure.
    fig.savefig(OUT / f"{dataset_name}_robustness.png", bbox_inches="tight")
    # I display the figure in the notebook.
    plt.show()
# I save the aggregated robustness results.
write_csv(OUT / "robustness_summary.csv", ROBUST_SUMMARY)

## 14. Visualizing learned filters and feature maps

Filters are trainable weights. Feature maps are the responses produced when the model processes a particular image. I use the first predefined training seed for these plots, rather than choosing the seed with the best result. For each dataset, I select three validation images with a fixed seed, without picking them based on the quality of their predictions.

I plot the initial and trained filters with the same color scale. Feature maps share a scale within each layer, but scales can differ between layers. A filter's appearance alone is not enough to conclude that it detects a particular edge or clothing category.

In [ ]:
# I use the first configured seed for visualisation rather than picking the best one.
VIS_SEED = CFG["seeds"][0]
# I show filters and feature maps for each dataset.
for dataset_name, data in DATA.items():
    # I get the main CNN's selected checkpoint for this seed.
    params = RUNS[(dataset_name, "cnn_max", VIS_SEED)]["params"]
    # I recreate the initial weights using the same seed.
    initial = init_model("cnn_max", VIS_SEED)
    # I extract the initial first-layer filters for the single input channel.
    initial_filters = np.asarray(initial["conv1"]["w"])[:, :, 0, :]
    # I extract the corresponding trained filters.
    trained_filters = np.asarray(params["conv1"]["w"])[:, :, 0, :]
    # I use one symmetric colour scale for both initial and trained weights.
    limit = float(max(np.abs(initial_filters).max(), np.abs(trained_filters).max()))
    # I create two rows of six filter panels.
    fig, axes = plt.subplots(2, 6, figsize=(10, 4))
    # I plot the initial filters in one row and the trained filters in the other.
    for row, filters in enumerate([initial_filters, trained_filters]):
        # I show each output channel's filter separately.
        for channel in range(6):
            # I display the filter weights using the shared colour scale.
            axes[row, channel].imshow(filters[:, :, channel], cmap="coolwarm", vmin=-limit, vmax=limit)
            # I label the filter number and whether it is initial or trained.
            axes[row, channel].set_title(("Initial " if row == 0 else "Trained ") + str(channel))
            # I hide the axes around each filter.
            axes[row, channel].axis("off")
    # I include the dataset and visualisation seed in the title.
    fig.suptitle(f"{dataset_name} | first-layer filters | seed={VIS_SEED}")
    # I adjust the spacing between filter panels.
    fig.tight_layout()
    # I save the filter comparison figure.
    fig.savefig(OUT / f"{dataset_name}_filters.png", bbox_inches="tight")
    # I display the filters in the notebook.
    plt.show()
    # I select three validation images with a fixed seed, without checking their predictions first.
    chosen_ids = np.random.default_rng(CFG["eval_seed"] + 55).choice(len(data["y_val"]), size=3, replace=False)
    # I save the selected indices so the same examples can be inspected again.
    (OUT / f"{dataset_name}_visualization_indices.json").write_text(json.dumps(chosen_ids.tolist()), encoding="utf-8")
    # I inspect each selected validation image.
    for sample_id in chosen_ids:
        # I select one image while keeping its batch dimension.
        image = data["x_val"][sample_id:sample_id + 1]
        # I get its class scores and intermediate feature maps.
        logits, features = forward(params, jnp.asarray(image), "cnn_max", return_features=True)
        # I select the class with the highest score.
        predicted = int(jnp.argmax(logits[0]))
        # I create a panel grid for the four intermediate stages.
        fig, axes = plt.subplots(4, 7, figsize=(12, 7))
        # I use one row for each intermediate feature-map stage.
        for row, (layer_name, feature) in enumerate(features.items()):
            # I convert this image's feature maps to NumPy for plotting.
            maps = np.asarray(feature[0])
            # I use a common intensity scale for the channels within this layer.
            vmax = max(float(maps.max()), 1e-8)
            # I repeat the input image in the first column as a reference.
            axes[row, 0].imshow(image[0, :, :, 0], cmap="gray", vmin=0, vmax=1)
            # I label the row with the layer name.
            axes[row, 0].set_title(layer_name, fontsize=8)
            # I display the first six channels from this layer.
            for channel in range(6):
                # I plot the channel's activation values using the layer's colour scale.
                axes[row, channel + 1].imshow(maps[:, :, channel], cmap="viridis", vmin=0, vmax=vmax)
                # I label the channel number.
                axes[row, channel + 1].set_title(f"channel {channel}", fontsize=8)
        # I go through all the panels to tidy their appearance.
        for ax in axes.ravel():
            # I hide the panel axes.
            ax.axis("off")
        # I show the dataset, image index, true label, and prediction in the title.
        fig.suptitle(f"{dataset_name} | validation index {sample_id} | true={data['y_val'][sample_id]} predicted={predicted}")
        # I adjust the feature-map layout.
        fig.tight_layout()
        # I save this image's feature-map figure.
        fig.savefig(OUT / f"{dataset_name}_feature_maps_{sample_id}.png", bbox_inches="tight")
        # I display the feature maps in the notebook.
        plt.show()

## 15. Main comparisons and interpretation

The next table calculates accuracy differences between model pairs using the same training seed, then averages those differences. A positive `accuracy_difference_pp` means the first model had higher accuracy. Results from three seeds on one fixed split do not establish that the same result will hold across every possible data split.

These are the questions I need to answer using the actual numbers and plots:

1. How did the CNN compare with the MLP on each dataset at a similar parameter count? How did their depth and computation differ?
2. How did max pooling, average pooling, and subsampling compare? Did their clean accuracy and robustness follow the same pattern?
3. What happened when I removed intermediate pooling in the GAP pair? The parameter count stayed fixed, but feature-map size and receptive field did not.
4. Did lower training loss lead to better validation performance, or were there signs of overfitting?
5. How did the filters and feature maps change from initialization? Which statements are visual observations rather than established explanations?
6. How do dataset size, epoch budget, shared settings, and the lack of model-specific tuning limit the conclusions, including any negative results?

In [ ]:
# I specify the four architecture comparisons before calculating the differences.
COMPARISONS = [("cnn_max", "mlp"), ("cnn_max", "cnn_avg"), ("cnn_max", "cnn_subsample"), ("cnn_max_gap", "cnn_no_pool_gap")]
# I prepare a table for the paired comparisons.
CONTRAST_ROWS = []
# I compare the models separately for each dataset.
for dataset_name in CFG["datasets"]:
    # I go through each predefined pair of models.
    for first, second in COMPARISONS:
        # I subtract their test accuracies using matching seeds and express the differences in percentage points.
        differences = [100 * (RUNS[(dataset_name, first, seed)]["summary"]["test_accuracy"] - RUNS[(dataset_name, second, seed)]["summary"]["test_accuracy"]) for seed in CFG["seeds"]]
        # I calculate the mean and standard deviation of the paired differences.
        average, spread = mean_std(differences)
        # I add the paired comparison to the table.
        CONTRAST_ROWS.append(dict(dataset=dataset_name, first_model=first, second_model=second, accuracy_difference_pp_mean=average, accuracy_difference_pp_std=spread, seeds=len(differences)))
        # I print the signed difference so it is clear which model performed better.
        print(f"{dataset_name}: {first} minus {second} = {average:+.2f} percentage points")
# I save the architecture comparisons as CSV.
write_csv(OUT / "paired_architecture_comparisons.csv", CONTRAST_ROWS)

## 16. Differences from the original LeNet-5

I use a **LeNet-style adaptation** for this project. I am not reproducing a particular figure or numerical result from the 1998 paper. Any later reproduction claim needs to identify the exact figure or experiment and document changes to the training budget and evaluation.

| Component | LeNet-5 in the paper | Implementation here |
|---|---|---|
| Input | 32×32 | 28×28, without extra padding |
| Feature path | C1(6), S2, C3(16), S4, C5(120), F6(84) | Conv6, Pool, Conv16, Pool, Dense120, Dense84 |
| C3 connectivity | Selected partial connections between feature maps | Full connectivity between input and output channels |
| Original subsampling | Aggregation with a trainable transformation | Parameter-free max or average pooling, plus a direct-subsampling control |
| Nonlinearity and output | Squashing nonlinearity and RBF outputs | ReLU and ten-class logits with cross-entropy |
| Data and training | The paper's protocol | The two datasets, split, and Adam settings documented here |

With the original 32×32 input, C5 is a convolution that produces 120 outputs. Here, I flatten the features and use Dense120. I need to include this architectural difference in the report.

### References

- LeCun, Bottou, Bengio & Haffner (1998), *Gradient-Based Learning Applied to Document Recognition*: https://leon.bottou.org/papers/lecun-98h
- JAX convolution primitive: https://docs.jax.dev/en/latest/_autosummary/jax.lax.conv_general_dilated.html
- JAX reduce-window primitive: https://docs.jax.dev/en/latest/_autosummary/jax.lax.reduce_window.html
- JAX automatic differentiation: https://docs.jax.dev/en/latest/automatic-differentiation.html
- Fashion-MNIST data and checksums: https://github.com/zalandoresearch/fashion-mnist
- MNIST data files: https://storage.googleapis.com/cvdf-datasets/mnist/
- Kingma & Ba, *Adam: A Method for Stochastic Optimization*: https://arxiv.org/abs/1412.6980
- Course Lab 2: the PyTorch examples covering MLPs, BatchNorm, Dropout, and CNNs provide a comparison point. This notebook uses a separate JAX implementation.

## 17. Saving the environment and packaging the results

This cell records the installed package versions, device, configuration, and output size. The requirements file records the Python packages used in the run. Reproducing a GPU run also needs a JAX installation that matches the environment's CUDA setup. Using the same seed does not guarantee bit-for-bit equality across CPUs, GPUs, or software versions.

After the final run, I still need to complete the four-page report, contribution statement, and short recorded presentation. My report outline is: page 1 for the question and architecture; page 2 for the protocol and correctness checks; page 3 for results and ablations; and page 4 for robustness, feature maps, limitations, contributions, and references. I write it in LaTeX and check the page count of the compiled PDF.

I keep the notebook in Git from the start and record each member's changes with actual commits. After downloading it, I can run these commands in the project folder. The notebook does not run them automatically.

```bash
# I create a local Git repository for the project.
git init
# I add the notebook to version control.
git add CNN_JAX_English.ipynb
# I commit the first version of the project.
git commit -m "Add JAX CNN implementation and controlled experiments"
```

In [ ]:
# I record the exact versions of the numerical and plotting packages.
VERSIONS = {name: metadata.version(name) for name in ["numpy", "matplotlib", "jax", "jaxlib"]}
# I collect the Python version, platform, devices, and run identifiers.
ENVIRONMENT = dict(python=sys.version, platform=platform.platform(), versions=VERSIONS, devices=[str(device) for device in jax.devices()], backend=jax.default_backend(), run_started_utc=RUN_STAMP, config_id=CONFIG_ID)
# I save the environment details as JSON.
(OUT / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
# I save the installed package versions in a requirements file.
(OUT / "requirements-recorded.txt").write_text("\n".join(f"{name}=={version}" for name, version in VERSIONS.items()) + "\n", encoding="utf-8")
# I prepare a short guide explaining how to reproduce and interpret the run.
REPRODUCTION = "Reproduction: open CNN_JAX_English.ipynb, use config.json settings, install the recorded dependencies in a compatible environment, and run cells in order. Dataset hashes, split indices, evaluation subsets and seeds are included. Best checkpoints are selected using validation loss only. Test results are not used for checkpoint selection. Train curves use a fixed training probe. Robustness uses a fixed test subset and identical perturbations across models. Timing includes compilation and validation. Quick-mode results are preliminary and must not be presented as full-data results. The notebook itself must be saved separately and committed to Git. Complete your own four-page LaTeX report, participant contribution statement and recorded presentation after analysing the real results. No report conclusion or presentation is fabricated by this notebook."
# I save the reproduction guide with the outputs.
(OUT / "REPRODUCE.txt").write_text(REPRODUCTION, encoding="utf-8")
# I create a blank contribution table for the team to complete with their actual work.
write_csv(OUT / "contributions_to_complete.csv", [{"participant": "", "implementation": "", "experiments": "", "analysis_and_report": "", "git_commits": ""}])
# I package the output folder into a ZIP file.
ZIP_PATH = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
# I print the path to the results ZIP.
print("Results ZIP:", ZIP_PATH)
# I print the ZIP size in MiB.
print("ZIP size (MiB):", round(Path(ZIP_PATH).stat().st_size / (1024 ** 2), 2))
# I print the folder containing the tables and figures.
print("Tables and figures:", OUT)

## 18. Final checklist

- [ ] All correctness checks pass.
- [ ] Both datasets and all core models have been run.
- [ ] Checkpoints were selected using validation, and the test set was kept out of tuning.
- [ ] Quick-mode results are clearly separated from full-mode results.
- [ ] I have reviewed parameter counts, accuracy tables, and learning curves.
- [ ] I have compared pooling with the same-size controls and the GAP pair.
- [ ] Translation and noise experiments use shared examples and perturbation seeds.
- [ ] I have explained the filters and feature maps, including the limits of visual interpretation.
- [ ] I have downloaded and versioned the results and the notebook itself.
- [ ] I have documented differences from the paper, team contributions, sources, and tool assistance as required by the course.
- [ ] The four-page LaTeX/PDF report and short recorded presentation are ready.

**Study notes:** To compare this notebook with Lab 2, I start with Section 5 on layers, then Section 6 on loss and Adam, and then Section 9 on training. I follow the array shapes as I read the JAX and PyTorch code side by side. The comments above each code line help me work through one part at a time.